In [1]:
import torch, torch.nn as nn

# 1. Vocabulary & Data Setup
text = "hello world, this is a simple text generation using LSTMs."
chars = sorted(list(set(text)))
c2i = {c: i for i, c in enumerate(chars)}
i2c = {i: c for i, c in enumerate(chars)}

X, y = [], []
for i in range(len(text) - 10):
    X.append([c2i[c] for c in text[i:i+10]])
    y.append(c2i[text[i+10]])
X_train, y_train = torch.tensor(X), torch.tensor(y)

# 2. Clean LSTM Architecture
class TextLSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, 16)
        self.lstm = nn.LSTM(16, 128, batch_first=True)
        self.fc = nn.Linear(128, vocab_size)

    def forward(self, x):
        _, (h, _) = self.lstm(self.embed(x))
        return self.fc(h[-1]) # h[-1] automatically gets the final state without tensor slicing

# 3. Training
model = TextLSTM(len(chars))
optimizer, criterion = torch.optim.Adam(model.parameters(), lr=0.01), nn.CrossEntropyLoss()

for epoch in range(50):
    optimizer.zero_grad()
    criterion(model(X_train), y_train).backward()
    optimizer.step()

# 4. Smart Text Generation
def generate(seed, length=50):
    model.eval()
    curr_seq = [c2i[c] for c in seed]
    for _ in range(length):
        with torch.no_grad():
            # Grabs the last 10 historical characters dynamically
            pred = model(torch.tensor([curr_seq[-10:]])).argmax(1).item() 
        seed += i2c[pred]
        curr_seq.append(pred)
    return seed

print(generate("hello world", 50))

hello world, this is a simple text generation using LSTMs.mt 
